# EDA Prediksi Risiko Penyakit Jantung — Heart Disease Dataset

Tugas Exploratory Data Analysis (EDA)
**Dataset:** `heart_disease_dataset.csv` (1.025 baris, 14 kolom)

Notebook ini berisi seluruh kode Python yang digunakan untuk melakukan Exploratory Data Analysis
(EDA) terhadap dataset penyakit jantung, mulai dari pengumpulan data, pembersihan data, analisis
deskriptif, hingga eksplorasi hubungan antar variabel. Hasil dari notebook ini menjadi dasar
pembahasan pada laporan makalah yang menyertainya.


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)


## 2. Pengumpulan Data

Dataset diperoleh dari penugasan mata kuliah berupa file `heart_disease_dataset.csv` yang berisi
1.025 baris data pasien dengan 14 kolom (13 fitur klinis + 1 label target `target`).

In [ ]:
df = pd.read_csv("heart_disease_dataset.csv")
print("Ukuran dataset:", df.shape)
df.head()


In [ ]:
df.info()


Deskripsi singkat setiap kolom pada dataset:

| Kolom | Deskripsi |
|---|---|
| age | Usia pasien (tahun) |
| sex | Jenis kelamin (1 = laki-laki, 0 = perempuan) |
| cp | Tipe nyeri dada (0-3) |
| trestbps | Tekanan darah istirahat (mm Hg) |
| chol | Kolesterol serum (mg/dl) |
| fbs | Gula darah puasa > 120 mg/dl (1 = ya, 0 = tidak) |
| restecg | Hasil elektrokardiografi istirahat (0-2) |
| thalach | Detak jantung maksimum tercapai |
| exang | Angina akibat olahraga (1 = ya, 0 = tidak) |
| oldpeak | Depresi ST akibat olahraga relatif terhadap istirahat |
| slope | Kemiringan segmen ST puncak latihan (0-2) |
| ca | Jumlah pembuluh utama yang terwarnai fluoroskopi (0-4) |
| thal | Kelainan thalassemia (1=normal, 2=cacat tetap, 3=cacat reversibel) |
| target | Label diagnosis (1 = terindikasi penyakit jantung, 0 = tidak) |


## 3. Pembersihan Data

In [ ]:
# Cek missing value
print("Total missing value:", df.isnull().sum().sum())
df.isnull().sum()


In [ ]:
# Cek data duplikat
n_dup = df.duplicated().sum()
print(f"Jumlah baris duplikat: {n_dup} dari {len(df)} baris ({n_dup/len(df)*100:.1f}%)")


Ditemukan sejumlah baris duplikat pada dataset. Untuk keperluan statistik yang tidak bias oleh
duplikasi (mis. distribusi & korelasi), kita siapkan versi data tanpa duplikat (`df_unique`) sebagai
pembanding, sementara analisis utama tetap memakai seluruh 1.025 baris sesuai instruksi tugas
(dataset diberikan apa adanya).

In [ ]:
df_unique = df.drop_duplicates().reset_index(drop=True)
print("Setelah drop duplicate:", df_unique.shape)


In [ ]:
# Cek tipe data & rentang nilai tiap kolom kategorikal untuk memastikan tidak ada nilai aneh/inkonsisten
cat_like = ["sex","cp","fbs","restecg","exang","slope","ca","thal","target"]
for c in cat_like:
    print(c, sorted(df[c].unique()))


## 4. Analisis Deskriptif

In [ ]:
df.describe().T


In [ ]:
target_counts = df["target"].value_counts()
print(target_counts)
print("Proporsi:", (target_counts / len(df) * 100).round(2).to_dict())

plt.figure(figsize=(5,4))
sns.countplot(x="target", hue="target", data=df, palette=["#4C72B0","#DD8452"], legend=False)
plt.title("Distribusi Kelas Target")
plt.xlabel("Target (0 = Tidak Sakit, 1 = Sakit)")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = ["age","trestbps","chol","thalach","oldpeak"]

fig, axes = plt.subplots(2,3, figsize=(15,8))
axes = axes.flatten()
for i,c in enumerate(numeric_cols):
    sns.histplot(df[c], kde=True, ax=axes[i], color="#4C72B0")
    axes[i].set_title(c)
axes[-1].axis("off")
plt.tight_layout()
plt.show()


### Deteksi Outlier (Metode IQR)

In [ ]:
outlier_summary = {}
for c in numeric_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df[c] < lower) | (df[c] > upper)).sum()
    outlier_summary[c] = n_out
outlier_summary


## 5. Exploratory Data Analysis (EDA)

### 5.1 Boxplot Variabel Numerik terhadap Target

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(15,8))
axes = axes.flatten()
for i,c in enumerate(numeric_cols):
    sns.boxplot(x="target", y=c, hue="target", data=df, ax=axes[i], palette=["#4C72B0","#DD8452"], legend=False)
    axes[i].set_title(f"{c} vs target")
axes[-1].axis("off")
plt.tight_layout()
plt.show()


### 5.2 Matriks Korelasi Pearson

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(11,9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, cbar_kws={"shrink":0.8})
plt.title("Matriks Korelasi Pearson Antar Variabel")
plt.tight_layout()
plt.show()


In [ ]:
corr["target"].sort_values(ascending=False)


### 5.3 Variabel Kategorikal terhadap Target

In [ ]:
cat_cols = ["sex","cp","fbs","restecg","exang","slope","ca","thal"]

fig, axes = plt.subplots(2,4, figsize=(18,9))
axes = axes.flatten()
for i,c in enumerate(cat_cols):
    ct = pd.crosstab(df[c], df["target"], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=axes[i], color=["#4C72B0","#DD8452"], legend=False)
    axes[i].set_title(c)
    axes[i].set_xlabel("")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, ["Target 0","Target 1"], loc="upper center", ncol=2)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


### 5.4 Hubungan Antar Variabel Numerik (Scatterplot)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x="age", y="thalach", hue="target", data=df, palette=["#4C72B0","#DD8452"], alpha=0.7)
plt.title("Hubungan Usia dan Detak Jantung Maksimum berdasarkan Target")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x="age", y="chol", hue="target", data=df, palette=["#4C72B0","#DD8452"], alpha=0.7)
plt.title("Hubungan Usia dan Kolesterol berdasarkan Target")
plt.tight_layout()
plt.show()


### 5.5 Pairplot Variabel Numerik Terpilih

In [ ]:
subset = df[["age","trestbps","chol","thalach","oldpeak","target"]]
g = sns.pairplot(subset, hue="target", palette=["#4C72B0","#DD8452"], diag_kind="hist", plot_kws={"alpha":0.6,"s":20})
g.fig.suptitle("Pairplot Variabel Numerik Terpilih", y=1.02)
plt.show()


### 5.6 Analisis Kelompok Usia terhadap Target

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[28,40,50,60,80], labels=["<40","40-49","50-59","60+"])
age_target = pd.crosstab(df["age_group"], df["target"], normalize="index") * 100
print(age_target.round(2))

age_target.plot(kind="bar", stacked=True, color=["#4C72B0","#DD8452"], figsize=(7,5))
plt.title("Proporsi Target berdasarkan Kelompok Usia")
plt.ylabel("Persentase (%)")
plt.xticks(rotation=0)
plt.legend(["Target 0","Target 1"])
plt.tight_layout()
plt.show()


## 6. Ringkasan Temuan

- Dataset berisi 1.025 baris tanpa *missing value*, namun mengandung 723 baris (± 70%) yang
  merupakan duplikat dari 302 baris unik — ciri khas dataset UCI Heart Disease versi replikasi.
- Kelas target relatif seimbang (51,3% positif vs 48,7% negatif) sehingga tidak diperlukan
  penanganan *imbalance* khusus.
- Variabel `cp` (tipe nyeri dada), `thalach` (detak jantung maksimum), dan `slope` berkorelasi
  positif dengan target, sedangkan `exang`, `oldpeak`, `ca`, dan `thal` berkorelasi negatif —
  konsisten dengan literatur medis mengenai indikator penyakit jantung.
- Pasien berusia di bawah 50 tahun pada dataset ini justru menunjukkan proporsi target positif
  yang lebih tinggi dibanding kelompok usia 60+, kemungkinan karena bias populasi sampel pada
  dataset ini.

Detail interpretasi lengkap dibahas pada laporan makalah (`.pdf`) yang menyertai notebook ini.
